## 캐싱(Caching)

LangChain은 LLM을 위한 선택적 캐싱 레이어를 제공합니다.

이는 두 가지 이유로 유용합니다.

- 동일한 완료를 여러 번 요청하는 경우 LLM 공급자에 대한 **API 호출 횟수를 줄여 비용을 절감**할 수 있습니다.
- LLM 제공업체에 대한 **API 호출 횟수를 줄여 애플리케이션의 속도를 높일 수** 있습니다.

### LLM 답변 캐싱하기 
- 답변 캐싱(Answer Caching)은 시스템 performance, 비용, 안정성을 모두 확보하기 위한 핵심 최적화 요소
- 동일하거나 유사한 질문에 대해 미리 저장된 결과를 반환함으로써 LLM API 비용을 0으로 만듬.
- 응답 속도(Latency) 획기적 개선
- 벡터 DB 검색 오버헤드 축소: 조회가 많을 경우.
- 답변 일관성 확보 : 유사한 질문에 대한 답변.

In [1]:
# !pip --version

pip 26.2.1 from D:\hykim\hanwha_0902\.venv\Lib\site-packages\pip (python 3.12)



In [2]:
# !pip install dotenv

In [23]:
!pip install langchain -U langchain-openai

  Attempting uninstall: langchain
    Found existing installation: langchain 1.4.0
    Uninstalling langchain-1.4.0:
      Successfully uninstalled langchain-1.4.0


In [3]:
from dotenv import load_dotenv

#.env파일에 설정된 보안정보를 읽기
load_dotenv()

True

In [4]:
!pip install langchain langchain_openai
!pip install langchain_community

  Using cached aiohttp-3.14.3-cp312-cp312-win_amd64.whl.metadata (8.5 kB)
  Using cached httpx_sse-0.4.3-py3-none-any.whl.metadata (9.7 kB)
  Using cached aiohappyeyeballs-2.7.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached frozenlist-1.8.0-cp312-cp312-win_amd64.whl.metadata (21 kB)
  Using cached multidict-6.8.0-cp312-cp312-win_amd64.whl.metadata (5.4 kB)
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ------------- -------------------------- 0.8/2.4 MB 4.8 MB/s eta 0:00:01
   ------------------------------- -------- 1.8/2.4 MB 4.8 MB/s eta 0:00:01
   ---------------------------------------- 2.4/2.4 MB 4.6 MB/s  0:00:00

   ---- ----------------------------------- 1/9 [multidict]
   ------------- -------------------------- 3/9 [frozenlist]
   ---------------------- ----------------- 5/9 [yarl]
   -------------------------- ------------- 6/9 [aiosignal]
   ------------------------------- -----

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate

In [6]:
llm = ChatOpenAI(model="gpt-3.5-turbo")

prompt = PromptTemplate.from_template("{country} 에 대해서 200자 내외로 요약해줘")

chain = prompt | llm

In [8]:
%%time    # %%time은 그 셀이 실행되는 데 걸린 시간을 잰 뒤 출력
response = chain.invoke({"country": "한국"})
print(response.content)
# 셀 명령어 : 셀(Cell) 전체의 실행 시간을 측정하는 셀 매직 명령어(Cell Magic Command)

한국은 동아시아에 위치한 대한민국과 조선민주주의인민공화국으로 나뉜다. 대한민국은 서구화된 고도의 경제력을 갖는 반면, 북한은 공산주의 체제를 유지하며 국제사회와의 접점이 제한된 상태이다. 한반도 전체를 아우르는 단일 국가 통일은 아직 이루어지지 않았지만, 남북 관계 개선을 위한 노력이 계속되고 있다. 한국은 고귀한 문화유산과 첨단 기술력을 보유한 나라로, K-pop, K-drama 등의 문화 콘텐츠가 세계적으로 인기를 끌고 있다. 한국은 빠르게 변화하는 현대화된 사회와 전통 문화를 조화롭게 이어받아 발전해 나가고 있다.
CPU times: total: 109 ms
Wall time: 4.89 s


셀 매직 명령어는 파이썬 문법이 아니라 Jupyter(IPython)가 제공하는 특수 명령이에요.

%% 두 개 = 셀 전체에 적용 (셀 매직). 반드시 셀 맨 첫 줄에 있어야 합니다.
% 한 개 = 그 줄 하나에만 적용 (라인 매직). 예: %time 함수()

주의 한 가지: .py 파일에서는 안 됩니다. %%time은 Jupyter 안에서만 동작해서, .ipynb를 .py로 옮기면 SyntaxError가 납니다. 그때는 time.time()으로 직접 재야 합니다.

## InMemoryCache

인메모리 캐시를 사용하여 동일 질문에 대한 답변을 저장하고, 캐시에 저장된 답변을 반환합니다.

In [ ]:
from langchain_core.globals import set_llm_cache
from langchain_core.caches import InMemoryCache

In [47]:
%%time

# 인메모리 캐시 사용
set_llm_cache(InMemoryCache())

response1 = chain.invoke({"country": "한국"})
print(response.content)

한국은 동아시아에 위치한 고대 역사와 현대 문화가 공존하는 나라이다. 국토는 70%가 산으로 이루어져 있고, 주요 도시인 수도 서울은 현대화된 도시로 발전하고 있다. 한국은 한반도 북쪽에 북한과 맞닿아 있어 분단국가로 알려져 있다. 한국은 전통적인 음식과 문화뿐만 아니라, K-pop, K-drama와 같은 대중문화로도 유명하다. 또한 최근에는 기술과 IT 분야에서도 세계적인 발전을 이루고 있다. 한반도의 산과 바다로 둘러싸인 자연환경과 선진 기술력과 문화가 공존하는 다채로운 매력을 가지고 있는 나라이다.
CPU times: total: 15.6 ms
Wall time: 2.98 s


## SQLite Cache

- 캐시(Cache) 레이어로 활용 : 질문(Query)의 임베딩 벡터나 텍스트 해시값을 키(Key)로, 생성된 LLM 답변 및 참고 문서 메타데이터를 값(Value)으로 저장하는 역학을 수행
- 빠름
- 영속성 : 순수 인메모리(In-Memory) 캐시와 달리 서버가 재시작되어도 캐시된 답변 데이터가 유실되지 않고 유지됨.

In [28]:
from langchain_community.cache import SQLiteCache  # 옛날 방식 (더 이상 권장되지 않음)
from langchain_core.globals import set_llm_cache
import os

In [44]:
# 캐시 디렉토리 생성
if not os.path.exists("cache"):     # 이 폴더/파일이 있나? → True/False
    os.makedirs("cache")            # 폴더 만들기 (이미 있으면 그냥 넘어감)

# SQLiteCache 사용
set_llm_cache(SQLiteCache(database_path="cache/llm_cache.db"))


In [43]:
%%time

response3 = chain.invoke({"country": "한국"})
print(response3.content)

한국은 동아시아에 위치한 고도 경제성장을 이룩한 선진국가이다. 전통과 현대가 공존하는 독특한 문화를 가지고 있으며 한류와 K-POP을 통해 세계적으로 큰 인기를 얻고 있다. 경제적으로는 세계 10대 경제국가 중 하나로 발전하였고, 기술력과 IT 산업에서 선두를 달리고 있다. 정치적으로는 대한민국으로 알려져 있으며 대통령제를 채택하고 있다. 한반도 북쪽의 북한과의 관계가 계속되는 도전과 과제이지만, 국내에서는 안정적이고 번영한 삶을 살고 있다. 한국은 아름다운 자연 환경과 고증적 건축물, 맛있고 다양한 음식 등으로 세계인의 관심을 끌고 있는 나라이다.
CPU times: total: 31.2 ms
Wall time: 11.2 ms


In [64]:
%%time

# SQLiteCache 사용
set_llm_cache(SQLiteCache())

response3 = chain.invoke({"country": "한국"})
print(response3.content)

한국은 동아시아에 위치한 독립국가로, 수도는 서울에 있다. 세계적으로 주목받는 문화, 음식, 기술 등 다양한 측면에서 높은 수준의 발전을 이루어왔다. 한류 열풍으로 인해 한국의 문화가 전 세계적으로 알려지면서 외국인 관광객도 점차 증가하고 있다. 현재는 선진화된 IT 산업과 자동차 산업을 중심으로 경제적으로도 안정적인 성장을 이어가고 있다. 한반도 분단 문제와 북한과의 관계는 여전히 국제사회의 이슈로 남아 있지만, 효율적인 외교정책과 미국 등 주요 국가들과의 협력을 통해 이 문제를 해결하려 노력하고 있다.
CPU times: total: 15.6 ms
Wall time: 16.4 ms


### 다른 확인 방법
- 지금 캐시: SQLiteCache   → Wall time 몇 ms 나와야 정상
- 지금 캐시: InMemoryCache → 셀 13을 다시 돌려서 바뀐 것. 셀 16 다시 실행

In [65]:
%%time
from langchain_core.globals import get_llm_cache
print("지금 캐시:", type(get_llm_cache()).__name__)

response3 = chain.invoke({"country": "한국"})
print(response3.content)


지금 캐시: SQLiteCache
한국은 동아시아에 위치한 독립국가로, 수도는 서울에 있다. 세계적으로 주목받는 문화, 음식, 기술 등 다양한 측면에서 높은 수준의 발전을 이루어왔다. 한류 열풍으로 인해 한국의 문화가 전 세계적으로 알려지면서 외국인 관광객도 점차 증가하고 있다. 현재는 선진화된 IT 산업과 자동차 산업을 중심으로 경제적으로도 안정적인 성장을 이어가고 있다. 한반도 분단 문제와 북한과의 관계는 여전히 국제사회의 이슈로 남아 있지만, 효율적인 외교정책과 미국 등 주요 국가들과의 협력을 통해 이 문제를 해결하려 노력하고 있다.
CPU times: total: 0 ns
Wall time: 9.97 ms
